# V18 — BMIA-V2 LR Sweep + Ablation (tab:v2_lr, tab:ablation_v2)
**Professor E**  
Goal: BMIA-V2 adds entropy filtering + λ_init=2.0 + gradient clipping over BMIA-A.  
Tab v2_lr: V2 accuracy across 4 LRs × 5 corruptions.  
Tab ablation_v2: isolate each of the 3 improvements.  
Output → `extreme_lr_mode.json`

**Kaggle**: Add Data → `hendrycks/cifar100c` | GPU T4 x2 | Internet ON | ~35 min

In [ ]:
import os,json,copy,gc,time
import numpy as np
import torch,torch.nn as nn,torch.optim as optim
import torchvision,torchvision.transforms as T
from torch.utils.data import DataLoader

torch.manual_seed(42); np.random.seed(42)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS=torch.cuda.device_count()
print(f'Device: {device}  GPUs: {N_GPUS}')

C100C_ROOT=None
for root,dirs,files in os.walk('/kaggle/input/'):
    if 'gaussian_noise.npy' in files and 'labels.npy' in files:
        C100C_ROOT=root; break
assert C100C_ROOT,'Add dataset: hendrycks/cifar100c'

NUM_CLASSES=100; TTA_BATCH=64; EVAL_BATCH=512
SEVERITY=5; SEV_OFFSET=(SEVERITY-1)*10000
CORRUPTIONS=['gaussian_noise','defocus_blur','snow','contrast','jpeg_compression']
LR_LIST=[0.005,0.01,0.02,0.05]
TAU=0.10
C_MEAN=torch.tensor([0.5071,0.4867,0.4408]).view(1,3,1,1)
C_STD =torch.tensor([0.2675,0.2565,0.2761]).view(1,3,1,1)

# BMIA-V2 improvements (ablation flags)
# ef=entropy_filter, lm=lambda_init=2.0, clip=grad_clip
ABLATION=[
    ('bmia_a_base',  False,False,False),  # original BMIA-A
    ('bmia_a_ef',    True, False,False),  # + entropy filter
    ('bmia_a_lm2',   False,True, False),  # + λ_init=2.0
    ('bmia_a_clip',  False,False,True ),  # + grad clip
    ('bmia_a_v2',    True, True, True ),  # V2 = all three
]
ABL_LR=0.05  # ablation at hardest LR

def load_c100c(c):
    data=np.load(f'{C100C_ROOT}/{c}.npy')[SEV_OFFSET:SEV_OFFSET+10000]
    lbl =np.load(f'{C100C_ROOT}/labels.npy')
    X=(torch.tensor(data).permute(0,3,1,2).float()/255.0-C_MEAN)/C_STD
    return X.to(device),torch.tensor(lbl,dtype=torch.long).to(device)

print(f'LR_LIST={LR_LIST}  ABL_LR={ABL_LR}')

In [ ]:
# ── Train ResNet-18 (3×3 conv1, 30 ep) ───────────────────────────
ds=torchvision.datasets.CIFAR100('/tmp',train=True,download=True,transform=T.Compose([
    T.RandomCrop(32,4),T.RandomHorizontalFlip(),T.ToTensor(),
    T.Normalize([0.5071,0.4867,0.4408],[0.2675,0.2565,0.2761])]))
dl=DataLoader(ds,batch_size=128,shuffle=True,num_workers=4,pin_memory=True)
model=torchvision.models.resnet18(weights=None)
model.conv1=nn.Conv2d(3,64,3,stride=1,padding=1,bias=False)
model.maxpool=nn.Identity()
model.fc=nn.Linear(512,NUM_CLASSES)
model=model.to(device)
dp=nn.DataParallel(model) if N_GPUS>1 else model
opt_tr=optim.SGD(model.parameters(),lr=0.1,momentum=0.9,weight_decay=5e-4)
sch=optim.lr_scheduler.CosineAnnealingLR(opt_tr,T_max=30)
crit=nn.CrossEntropyLoss()
dp.train(); t0=time.time()
for ep in range(30):
    for x,y in dl:
        x,y=x.to(device),y.to(device)
        opt_tr.zero_grad(); crit(dp(x),y).backward(); opt_tr.step()
    sch.step()
    if (ep+1)%10==0: print(f'  ep={ep+1}/30  {(time.time()-t0)/60:.1f}min')
model.eval(); SOURCE_STATE=copy.deepcopy(model.state_dict())
del dp,dl,ds; gc.collect(); torch.cuda.empty_cache()
print('Training done.')

In [ ]:
# ── run_bmia with ablation flags ─────────────────────────────────
def freeze_bn(mdl):
    ids={id(p) for m in mdl.modules() if isinstance(m,nn.BatchNorm2d) for p in m.parameters()}
    for p in mdl.parameters(): p.requires_grad_(id(p) in ids)

def get_acc(mdl,X,y):
    mdl.eval(); preds=[]
    with torch.no_grad():
        for i in range(0,X.size(0),EVAL_BATCH): preds.append(mdl(X[i:i+EVAL_BATCH]).argmax(1))
    p=torch.cat(preds)
    nc=(torch.bincount(p,minlength=NUM_CLASSES)>0).sum().item()
    dom=torch.bincount(p,minlength=NUM_CLASSES).float().max().item()/p.size(0)
    return round((p==y[:len(p)]).float().mean().item(),4),round(dom,4),nc

def run_bmia(X,y,lr,use_ef=False,use_lm=False,use_clip=False):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train(); freeze_bn(model)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=optim.SGD(params,lr=lr,momentum=0.9)
    init={pn:p.data.clone() for pn,p in model.named_parameters() if p.requires_grad}
    lam=2.0 if use_lm else 0.5

    for i in range(0,X.size(0),TTA_BATCH):
        xb=X[i:i+TTA_BATCH]
        if xb.size(0)<4: continue
        opt.zero_grad()
        pr=torch.softmax(model(xb),1); ent=-(pr*torch.log(pr+1e-8)).sum(1)

        if use_ef:
            mask=ent<0.4*np.log(NUM_CLASSES)
            if mask.sum()<2: continue
            pr_f=pr[mask]; ent_f=ent[mask]
        else:
            pr_f=pr; ent_f=ent

        mg=pr_f.mean(0); hy=-(mg*torch.log(mg+1e-8)).sum()
        anc=sum(((p-init[pn])**2).sum() for pn,p in model.named_parameters() if pn in init)
        loss=ent_f.mean()-hy+lam*anc
        loss.backward()

        if use_clip: torch.nn.utils.clip_grad_norm_(params,max_norm=1.0)
        opt.step()

        with torch.no_grad():
            dom=(torch.bincount(pr.argmax(1),minlength=NUM_CLASSES).float().max()/xb.size(0)).item()
        lam=min(10.,lam*1.1) if dom>TAU else max(0.01,lam*0.95)

    return get_acc(model,X,y)

print('run_bmia ready. Ablation variants:', [v[0] for v in ABLATION])

In [ ]:
# ── Tab v2_lr: V2 vs V1 across LRs ──────────────────────────────
lr_results=[]; t0=time.time()
for corr in CORRUPTIONS:
    X,y=load_c100c(corr); print(f'\n=== {corr} ===')
    for lr in LR_LIST:
        a1,d1,c1=run_bmia(X,y,lr,False,False,False)  # V1
        a2,d2,c2=run_bmia(X,y,lr,True, True, True )  # V2
        col1=(d1>0.15 and c1<50) or c1<20
        col2=(d2>0.15 and c2<50) or c2<20
        print(f'  lr={lr}  V1={a1*100:.1f}% [{"COL" if col1 else "ok"}]  '
              f'V2={a2*100:.1f}% [{"COL" if col2 else "ok"}]  gain={+(a2-a1)*100:.1f}%')
        lr_results.append({'corruption':corr,'lr':lr,'version':'bmia_adaptive',
                           'acc':a1,'dom_pct':d1,'n_classes':c1,'collapsed':col1})
        lr_results.append({'corruption':corr,'lr':lr,'version':'bmia_adaptive_v2',
                           'acc':a2,'dom_pct':d2,'n_classes':c2,'collapsed':col2})
        gc.collect(); torch.cuda.empty_cache()
    del X,y; gc.collect(); torch.cuda.empty_cache()
print(f'LR sweep done {(time.time()-t0)/60:.1f}min')

# ── Tab ablation_v2: each improvement at lr=0.05 ─────────────────
abl_results=[]
for corr in CORRUPTIONS:
    X,y=load_c100c(corr); print(f'\nAblation {corr}:')
    for name,ef,lm,clip in ABLATION:
        acc,dom,nc=run_bmia(X,y,ABL_LR,ef,lm,clip)
        col=(dom>0.15 and nc<50) or nc<20
        abl_results.append({'corruption':corr,'variant':name,'lr':ABL_LR,
                            'use_ef':ef,'use_lm':lm,'use_clip':clip,
                            'acc':acc,'dom_pct':dom,'n_classes':nc,'collapsed':col})
        print(f'  {name:<18} acc={acc*100:.1f}%  {"COLLAPSED" if col else "ok"}')
        gc.collect(); torch.cuda.empty_cache()
    del X,y; gc.collect(); torch.cuda.empty_cache()

# ── Print summary ─────────────────────────────────────────────────
print('\n=== V2 vs V1 avg accuracy ===')
for lr in LR_LIST:
    v1=np.mean([r['acc'] for r in lr_results if r['version']=='bmia_adaptive' and r['lr']==lr])*100
    v2=np.mean([r['acc'] for r in lr_results if r['version']=='bmia_adaptive_v2' and r['lr']==lr])*100
    print(f'  lr={lr}:  V1={v1:.1f}%  V2={v2:.1f}%  gain=+{v2-v1:.1f}%')

print('\n=== Ablation at lr=0.05 ===')
for name,ef,lm,clip in ABLATION:
    avg=np.mean([r['acc'] for r in abl_results if r['variant']==name])*100
    nc =sum(1 for r in abl_results if r['variant']==name and r['collapsed'])
    print(f'  {name:<18} avg={avg:.1f}%  collapses={nc}/{len(CORRUPTIONS)}')

with open('extreme_lr_mode.json','w') as f:
    json.dump({'experiment':'extreme_lr_mode','lr_list':LR_LIST,'abl_lr':ABL_LR,
               'severity':SEVERITY,'corruptions':CORRUPTIONS,
               'lr_results':lr_results,'ablation_results':abl_results},f,indent=2)
print('Saved → extreme_lr_mode.json')
with open('extreme_lr_mode.json') as f: print(f.read())